# 02 · Feature matching

**Goal:** check that descriptor matches connect the same physical points, choose the ratio-test threshold, and see which images are well connected before geometric verification (Phase 4).

Set `SCENE_DIR` (or the `MV3D_SCENE` environment variable) and `PAIR`, then run all cells. Features come from the on-disk cache when available.

In [ ]:
%matplotlib inline
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from src.config import FeatureConfig, load_config
from src.data.dataset import prepare_dataset
from src.features.extractor import FeatureCache, create_extractor, extract_features
from src.features.matcher import (
    DescriptorMatcher,
    connectivity,
    match_matrix,
    match_pairs,
    select_pairs,
)
from src.visualization.matches import draw_matches, plot_match_matrix

SCENE_DIR = Path(os.environ.get('MV3D_SCENE', '../data/raw/sample_scene'))
CONFIG_PATH = None  # None -> configs/default.yaml
PAIR = (0, 1)  # image indices to inspect in detail

# Notebooks run from notebooks/: keep caches in the repository's outputs/cache.
config = load_config(CONFIG_PATH).with_overrides({'output.cache_dir': '../outputs/cache'})
cache_dir = config.cache_path('features')
cache = FeatureCache(cache_dir) if cache_dir else None
SCENE_DIR.resolve()

## 1. Features

In [ ]:
dataset = prepare_dataset(SCENE_DIR, config)
names = [im.name for im in dataset.images]
extractor = create_extractor(config.features)
features, feature_stats = extract_features(dataset.images, extractor, cache)
pd.DataFrame([vars(s) for s in feature_stats]).set_index('image').round(3)

## 2. Match graph

Match count for every pair after the ratio test and mutual check. A bright band next to the diagonal means consecutive photos overlap; a dark row means an image matches nothing and will not be registered.

In [ ]:
pairs = select_pairs(len(features), config.matching)
matcher = DescriptorMatcher(config.matching)
matches, stats = match_pairs(features, pairs, matcher)
display(plot_match_matrix(match_matrix(stats, len(names)), names))

best = connectivity(stats, len(names))
isolated = [names[k] for k in np.flatnonzero(best < config.matching.min_matches)]
print(f'{len(pairs)} pairs, {sum(s.time_ms for s in stats) / 1000:.1f} s')
print('isolated images:', isolated or 'none')

## 3. Matches for one pair

Lines should connect the same scene point. Long lines that cross the general flow are wrong matches; Phase 4's geometric verification is meant to remove them.

In [ ]:
i, j = PAIR
m = next(x for x in matches if (x.i, x.j) == (i, j))
canvas = draw_matches(
    dataset.images[i].image, dataset.images[j].image, features[i], features[j], m
)
fig, ax = plt.subplots(figsize=(14, 14 * canvas.shape[0] / canvas.shape[1]))
ax.imshow(canvas[..., ::-1])
ax.set_title(f'{names[i]} - {names[j]}: {len(m)} matches (200 drawn)')
ax.axis('off');

## 4. Choosing the ratio threshold

Lowe ratio $d_1/d_2$ for every feature of image *i*. Correct matches have a clearly better nearest neighbour (low ratio); ambiguous ones sit near 1. The threshold should sit in the valley between the two groups. The right plot shows how many matches survive each threshold.

In [ ]:
ratios = matcher.ratios(features[i], features[j])
thresholds = np.round(np.arange(0.5, 0.96, 0.05), 2)
rows = []
for r in thresholds:
    _, s = DescriptorMatcher(
        type(config.matching)(**{**vars(config.matching), 'ratio': float(r)})
    ).match(features[i], features[j], i, j)
    rows.append({'ratio': r, 'after_ratio': s.after_ratio, 'after_mutual': s.final})
sweep = pd.DataFrame(rows).set_index('ratio')

fig, (a, b) = plt.subplots(1, 2, figsize=(12, 3.5))
a.hist(ratios, bins=50, color='tab:blue')
a.axvline(config.matching.ratio, color='k', ls='--', label=f'ratio {config.matching.ratio}')
a.set_xlabel('d1 / d2')
a.set_ylabel('features')
a.legend()
sweep.plot(ax=b, marker='o')
b.axvline(config.matching.ratio, color='k', ls='--')
b.set_ylabel('matches')
fig.tight_layout()
sweep.T

## 5. SIFT vs ORB on this pair

A preview of experiment E4 (RQ4): same images, same matching settings.

In [ ]:
rows = []
for detector in ['sift', 'orb']:
    cfg = FeatureConfig(detector=detector, max_features=config.features.max_features)
    ex = create_extractor(cfg)
    t0 = time.perf_counter()
    fi, fj = ex.detect(dataset.images[i].gray), ex.detect(dataset.images[j].gray)
    t_detect = (time.perf_counter() - t0) * 1000 / 2
    _, s = DescriptorMatcher(config.matching).match(fi, fj, i, j)
    rows.append({'detector': detector, 'features_i': len(fi), 'features_j': len(fj),
                 'after_ratio': s.after_ratio, 'matches': s.final,
                 'detect_ms_per_image': round(t_detect), 'match_ms': round(s.time_ms)})
pd.DataFrame(rows).set_index('detector')

## 6. Findings

Record before Phase 4:

- Isolated images and why (blur, no overlap, unrelated):
- Do the drawn matches look correct? Obvious wrong matches:
- Where the ratio histogram's valley is, and the chosen `matching.ratio`:
- SIFT vs ORB match counts on this scene: